In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sps
import scipy.special as sp_spec
import scipy.stats as sp_stats
import tqdm

## Assignment 2E, 2026 - LDA with SVI (Section 2.2)
In this assignment you will implement Stochastic Variational Inference (SVI) for Latent Dirichlet Allocation (LDA), using the provided Coordinate Ascent Variational Inference (CAVI) updates as a template.

Your SVI implementation should build on the CAVI implementation. You need to perform the local updates on a mini-batch of documents and then update the global parameters with a natural-gradient step (Hoffman et al., 2013, Figure 6).

For the simulated datasets (Cases 1–3), do not expect the expectations to be identical to the "true" $\theta$ and $\beta$. The focus is on implementing SVI correctly and comparing its convergence and runtime with CAVI.
In Case 4 you apply a **bag-of-words** version of SVI to a real corpus (20 Newsgroups), where documents have different lengths.

**Conventions.** In Cases 1–3 each document has the same length $N$ and the words are stored one-hot, `w[d, n]` $\in\{0,1\}^W$. One *epoch* of SVI is $\lfloor D/S\rfloor$ mini-batch steps, i.e. roughly one pass over the data, just like one CAVI iteration. The ELBO is recorded once per epoch (per iteration for CAVI), and runtimes exclude ELBO evaluation. The evaluation plots show the ELBO both per pass over the data and against training time.

### Generate data
The cells below generate data from the LDA model. For simplicity we use $N_d = N$ for all $d$ in the simulated cases.

In [ ]:
def generate_data(D, N, K, W, eta, alpha, rng):
    # Sample from the LDA generative model. Returns one-hot words w [D, N, W], topics z [D, N],
    # document-topic proportions theta [D, K] and topic-word distributions beta [K, W].
    beta = rng.dirichlet(eta, size=K)                        # K x W
    theta = rng.dirichlet(alpha, size=D)                     # D x K
    # sample z and w by inverse-CDF sampling (vectorised over documents and words)
    u = rng.random((D, N))
    z = (u[..., None] > np.cumsum(theta, axis=1)[:, None, :]).sum(axis=-1)
    z = np.minimum(z, K - 1)
    u = rng.random((D, N))
    w_label = np.zeros((D, N), dtype=int)
    for k in range(K):                                       # words of topic k via the topic's CDF
        mask = z == k
        w_label[mask] = np.searchsorted(np.cumsum(beta[k]), u[mask], side="right")
    w_label = np.minimum(w_label, W - 1)
    w = np.zeros((D, N, W), dtype=np.float32)
    np.put_along_axis(w, w_label[..., None], 1.0, axis=-1)
    return w, z, theta, beta


rng_sim = np.random.default_rng(0)  # Do not change the seed
D_sim, N_sim, K_sim, W_sim = 500, 50, 2, 5
eta_sim = np.ones(W_sim)
eta_sim[3] = 0.0001  # Expect word 3 to (almost) never appear in the data
eta_sim[1] = 3.      # Expect word 1 to be the most common word
alpha_sim = np.ones(K_sim)
w0, z0, theta0, beta0 = generate_data(D_sim, N_sim, K_sim, W_sim, eta_sim, alpha_sim, rng_sim)
w_cat = w0.argmax(axis=-1)
unique_z, counts_z = np.unique(z0[0, :], return_counts=True)
unique_w, counts_w = np.unique(w_cat[0, :], return_counts=True)

# Sanity checks for data generation
print(f"Average z of each document should be close to theta of document. \n Theta of doc 0: {theta0[0]} \n Mean z of doc 0: {counts_z / N_sim}")
print(f"Beta of topic 0: {beta0[0]}")
print(f"Beta of topic 1: {beta0[1]}")
print(f"Word to topic assignment, z, of document 0: {z0[0, 0:10]}")
print(f"Observed words, w, of document 0: {w_cat[0, 0:10]}")
print(f"Unique words and count of document 0: {[f'{u}: {c}' for u, c in zip(unique_w, counts_w)]}")

### Helper functions

In [ ]:
def log_multivariate_beta_function(a, axis=None):
    return np.sum(sp_spec.gammaln(a), axis=axis) - sp_spec.gammaln(np.sum(a, axis=axis))


def expected_log_dirichlet(a):
    # E[log x] under Dir(a), row-wise: digamma(a) - digamma(sum(a))
    return sp_spec.digamma(a) - sp_spec.digamma(np.sum(a, axis=-1, keepdims=True))

### CAVI implementation, ELBO and initialisation (provided)

In [ ]:
def initialize_q(w, D, N, K, W, rng):
    # Random initialisation.
    phi_init = rng.random(size=(D, N, K))
    phi_init = phi_init / np.sum(phi_init, axis=-1, keepdims=True)
    gamma_init = rng.integers(1, 10, size=(D, K)).astype(float)
    lmbda_init = rng.integers(1, 10, size=(K, W)).astype(float)
    return phi_init, gamma_init, lmbda_init


def update_q_Z(w, gamma, lmbda):
    # phi_dnk ∝ exp(E[log theta_dk] + E[log beta_{k, w_dn}])
    E_log_theta = expected_log_dirichlet(gamma)                          # D x K
    E_log_beta = expected_log_dirichlet(lmbda)                           # K x W
    w_label = w.argmax(axis=-1)                                          # D x N (index of the one-hot word)
    log_rho = E_log_theta[:, None, :] + E_log_beta.T[w_label]            # D x N x K
    return np.exp(log_rho - sp_spec.logsumexp(log_rho, axis=-1, keepdims=True))


def update_q_theta(phi, alpha):
    # gamma_d = alpha + sum_n phi_dn
    return alpha + np.sum(phi, axis=1)


def update_q_beta(w, phi, eta):
    # lambda_k = eta + sum_d sum_n phi_dnk w_dn
    return eta + np.einsum("dnk,dnw->kw", phi, w, optimize=True)


def calculate_elbo(w, phi, gamma, lmbda, eta, alpha):
    D, N, K = phi.shape
    W = eta.shape[0]
    E_log_theta = expected_log_dirichlet(gamma)                          # D x K
    E_log_beta = expected_log_dirichlet(lmbda)                           # K x W
    w_label = w.argmax(axis=-1)

    neg_CE_likelihood = np.sum(phi * E_log_beta.T[w_label])
    neg_CE_Z = np.einsum("dnk,dk->", phi, E_log_theta)
    neg_CE_theta = -D * log_multivariate_beta_function(alpha) + np.einsum("k,dk->", alpha - 1, E_log_theta)
    neg_CE_beta = -K * log_multivariate_beta_function(eta) + np.einsum("w,kw->", eta - 1, E_log_beta)
    H_Z = -np.sum(sp_spec.xlogy(phi, phi))
    gamma_0 = np.sum(gamma, axis=1)
    H_theta = np.sum(log_multivariate_beta_function(gamma, axis=1) + (gamma_0 - K) * sp_spec.digamma(gamma_0)
                     - np.sum((gamma - 1) * sp_spec.digamma(gamma), axis=1))
    lmbda_0 = np.sum(lmbda, axis=1)
    H_beta = np.sum(log_multivariate_beta_function(lmbda, axis=1) + (lmbda_0 - W) * sp_spec.digamma(lmbda_0)
                    - np.sum((lmbda - 1) * sp_spec.digamma(lmbda), axis=1))
    return neg_CE_likelihood + neg_CE_Z + neg_CE_theta + neg_CE_beta + H_Z + H_theta + H_beta


def CAVI_algorithm(w, K, n_iter, eta, alpha, rng, compute_elbo=True):
    # Returns final (phi, gamma, lmbda), the ELBO per iteration, and the cumulative runtime after each
    # iteration (excluding ELBO evaluation).
    D, N, W = w.shape
    phi, gamma, lmbda = initialize_q(w, D, N, K, W, rng)
    elbo, times = [], []
    train_time = 0.0
    for i in tqdm.tqdm(range(n_iter), desc="CAVI"):
        t0 = time.time()
        phi = update_q_Z(w, gamma, lmbda)        # q(Z) update
        gamma = update_q_theta(phi, alpha)       # q(theta) update
        lmbda = update_q_beta(w, phi, eta)       # q(beta) update
        train_time += time.time() - t0
        times.append(train_time)
        if compute_elbo:
            elbo.append(calculate_elbo(w, phi, gamma, lmbda, eta, alpha))
    return phi, gamma, lmbda, np.array(elbo), np.array(times)


rng_cavi = np.random.default_rng(1)  # Do not change the seed
phi_cavi0, gamma_cavi0, lmbda_cavi0, elbo_cavi0, times_cavi0 = CAVI_algorithm(
    w0, K_sim, 100, np.ones(W_sim), np.ones(K_sim), rng_cavi)

In [ ]:
precision = 3
print("----- Recall label switching - compare E[theta] and true theta and check for label switching -----")
print(f"Final E[theta] of doc 0 CAVI:  {np.round(gamma_cavi0[0] / np.sum(gamma_cavi0[0]), precision)}")
print(f"True theta of doc 0:           {np.round(theta0[0], precision)}")
print("----- Recall label switching - e.g. E[beta_0] could be fit to true beta_1. -----")
for k in range(K_sim):
    print(f"Final E[beta] k={k}: {np.round(lmbda_cavi0[k] / np.sum(lmbda_cavi0[k]), precision)}")
for k in range(K_sim):
    print(f"True beta k={k}:     {np.round(beta0[k], precision)}")

### ***Question 2.2.7: SVI implementation***

Using the CAVI updates as a template, finish the code below. `batch` is an array of $S$ distinct document indices.
- `update_q_Z_svi` and `update_q_theta_svi` perform the *local* updates for the documents in the batch only.
- `update_q_beta_svi` returns the *intermediate* global parameter $\hat\lambda = \eta + \frac{D}{S}\sum_{d\in\text{batch}}\sum_n \phi_{dn} w_{dn}^\top$, i.e. the CAVI update computed as if the batch were replicated $D/S$ times.
- `SVI_algorithm` iterates the local updates on the batch until convergence and then takes the natural-gradient step $\lambda\leftarrow(1-\rho_t)\lambda+\rho_t\hat\lambda$ with $\rho_t=(t+\tau)^{-\kappa}$.

In [ ]:
def update_q_Z_svi(batch, w, gamma, lmbda):
    # Return phi for the documents in the batch, shape [S, N, K].
    # TODO: rewrite the CAVI update to an SVI update
    return phi_batch


def update_q_theta_svi(batch, phi, alpha):
    # Return gamma for the documents in the batch, shape [S, K].
    # TODO: rewrite the CAVI update to an SVI update
    return gamma_batch


def update_q_beta_svi(batch, w, phi, eta):
    # Return the intermediate global parameter lambda_hat, shape [K, W].
    # TODO: rewrite the CAVI update to an SVI update
    return lmbda_hat


def SVI_algorithm(w, K, S, n_epochs, eta, alpha, rng, tau=1.0, kappa=0.9, compute_elbo=True,
                  max_local_iter=100, local_tol=1e-2):
    # Returns final (phi, gamma, lmbda), the ELBO per epoch, and the cumulative runtime after each epoch
    # (excluding ELBO evaluation).
    D, N, W = w.shape
    phi, gamma, lmbda = initialize_q(w, D, N, K, W, rng)
    steps_per_epoch = D // S
    elbo, times = [], []
    train_time = 0.0
    t = 0
    for epoch in tqdm.tqdm(range(n_epochs), desc="SVI"):
        t0 = time.time()
        for _ in range(steps_per_epoch):
            rho = (t + tau) ** (-kappa)                          # step size
            batch = rng.choice(D, size=S, replace=False)         # mini-batch of distinct documents
            gamma[batch] = 1.0                                   # re-initialise local parameters
            # TODO: update the local variational parameters (phi, gamma) of the batch until convergence
            
            # TODO: update the global variational parameters (lmbda)
            
            t += 1
        train_time += time.time() - t0
        times.append(train_time)
        if compute_elbo:
            # evaluation (not timed): refit the local parameters of all documents for the current lambda
            phi_eval, gamma_eval = phi.copy(), np.ones_like(gamma)
            for _ in range(max_local_iter):
                gamma_prev = gamma_eval
                phi_eval = update_q_Z(w, gamma_eval, lmbda)
                gamma_eval = update_q_theta(phi_eval, alpha)
                if np.mean(np.abs(gamma_eval - gamma_prev)) < local_tol:
                    break
            elbo.append(calculate_elbo(w, phi_eval, gamma_eval, lmbda, eta, alpha))
    return phi, gamma, lmbda, np.array(elbo), np.array(times)

In [ ]:
def compare_runs(theta_true, beta_true, gamma_cavi, lmbda_cavi, gamma_svi, lmbda_svi,
                 elbo_cavi, elbo_svi, times_cavi, times_svi, title):
    # Provided evaluation: expectations, ELBO traces and runtimes.
    np.set_printoptions(formatter={"float": lambda x: f"{x:0.3f}"})
    print("----- Recall label switching - compare E[theta] and true theta -----")
    print(f"E[theta] of doc 0 SVI:  {gamma_svi[0] / gamma_svi[0].sum()}")
    print(f"E[theta] of doc 0 CAVI: {gamma_cavi[0] / gamma_cavi[0].sum()}")
    print(f"True theta of doc 0:    {theta_true[0]}")
    for k in range(beta_true.shape[0]):
        print(f"E[beta] SVI k={k}:  {lmbda_svi[k] / lmbda_svi[k].sum()}")
    for k in range(beta_true.shape[0]):
        print(f"E[beta] CAVI k={k}: {lmbda_cavi[k] / lmbda_cavi[k].sum()}")
    for k in range(beta_true.shape[0]):
        print(f"True beta k={k}:    {beta_true[k]}")
    print(f"Final ELBO  CAVI: {elbo_cavi[-1]:.2f}   SVI: {elbo_svi[-1]:.2f}   difference: {elbo_svi[-1] - elbo_cavi[-1]:.2f}")
    print(f"Runtime     CAVI: {times_cavi[-1]:.2f} s ({len(times_cavi)} iterations)   "
          f"SVI: {times_svi[-1]:.2f} s ({len(times_svi)} epochs)")
    lo = min(elbo_cavi.min(), elbo_svi.min())
    hi = max(elbo_cavi.max(), elbo_svi.max())
    ylim = (hi - 0.05 * (hi - lo) - 0.02 * abs(hi), hi + 0.01 * (hi - lo))
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    axs[0].plot(np.arange(1, len(elbo_cavi) + 1), elbo_cavi, label="CAVI (per iteration)")
    axs[0].plot(np.arange(1, len(elbo_svi) + 1), elbo_svi, label="SVI (per epoch)")
    axs[0].set_xlabel("iteration / epoch (passes over the data)")
    axs[1].plot(times_cavi, elbo_cavi, label="CAVI")
    axs[1].plot(times_svi, elbo_svi, label="SVI")
    axs[1].set_xlabel("training time (s)")
    for ax in axs:
        ax.set_ylabel("ELBO"); ax.set_title(f"ELBO, {title}"); ax.legend(); ax.set_ylim(*ylim)
    plt.tight_layout()
    plt.show()

### CASE 1

In [ ]:
rng1 = np.random.default_rng(1)  # Do not change the seed
D1, N1, K1, W1 = 50, 50, 2, 5
w1, z1, theta1, beta1 = generate_data(D1, N1, K1, W1, np.ones(W1), np.ones(K1), rng1)

eta_prior1, alpha_prior1 = np.ones(W1), np.ones(K1)
S1 = D1 // 10  # batch size (documents)
_, gamma1_cavi, lmbda1_cavi, elbo1_cavi, time1_cavi = CAVI_algorithm(w1, K1, 100, eta_prior1, alpha_prior1, rng1)
_, gamma1_svi, lmbda1_svi, elbo1_svi, time1_svi = SVI_algorithm(w1, K1, S1, 50, eta_prior1, alpha_prior1, rng1)

#### Evaluation
Do not expect the expectations to be identical to the "true" $\theta$ and $\beta$, and do not expect the SVI ELBO curve to match the CAVI curve. It should, however, increase and end up in the same ballpark as CAVI.

In [ ]:
compare_runs(theta1, beta1, gamma1_cavi, lmbda1_cavi, gamma1_svi, lmbda1_svi,
             elbo1_cavi, elbo1_svi, time1_cavi, time1_svi, "Case 1")

### CASE 2

In [ ]:
rng2 = np.random.default_rng(2)  # Do not change the seed
D2, N2, K2, W2 = 1000, 50, 3, 10
w2, z2, theta2, beta2 = generate_data(D2, N2, K2, W2, np.ones(W2), np.ones(K2), rng2)

eta_prior2, alpha_prior2 = np.ones(W2), np.ones(K2)
S2 = D2 // 10  # batch size (documents)
_, gamma2_cavi, lmbda2_cavi, elbo2_cavi, time2_cavi = CAVI_algorithm(w2, K2, 100, eta_prior2, alpha_prior2, rng2)
_, gamma2_svi, lmbda2_svi, elbo2_svi, time2_svi = SVI_algorithm(w2, K2, S2, 20, eta_prior2, alpha_prior2, rng2)

In [ ]:
compare_runs(theta2, beta2, gamma2_cavi, lmbda2_cavi, gamma2_svi, lmbda2_svi,
             elbo2_cavi, elbo2_svi, time2_cavi, time2_svi, "Case 2")

### CASE 3
A larger dataset: one CAVI iteration vs. one SVI epoch, for a runtime comparison (no ELBO).

In [ ]:
rng3 = np.random.default_rng(3)  # Do not change the seed
D3, N3, K3, W3 = 1000, 1000, 10, 100
w3, z3, theta3, beta3 = generate_data(D3, N3, K3, W3, np.ones(W3), np.ones(K3), rng3)

eta_prior3, alpha_prior3 = np.ones(W3), np.ones(K3)
S3 = D3 // 10  # batch size (documents)
*_, times3_cavi = CAVI_algorithm(w3, K3, 1, eta_prior3, alpha_prior3, rng3, compute_elbo=False)
*_, times3_svi = SVI_algorithm(w3, K3, S3, 1, eta_prior3, alpha_prior3, rng3, compute_elbo=False)
print(f"Runtime of one pass over the data: CAVI {times3_cavi[-1]:.2f} s, SVI {times3_svi[-1]:.2f} s")
del w3  # free memory

*Your comments here.*

### CASE 4: 20 Newsgroups with a bag-of-words SVI (***Question 2.2.8***)

Real documents have different lengths $N_d$, and a one-hot array $[D, N_{\max}, W]$ would not fit in memory. In the **bag-of-words** representation a corpus is a sparse count matrix $X\in\mathbb N^{D\times W}$, with $x_{dw}$ = the number of times word $w$ occurs in document $d$. Because all occurrences of the same word in a document share the same variational distribution $\phi_{dw}$, the LDA updates become sums over the *distinct* words of each document, weighted by their counts:
$$
\phi_{dwk}\propto\exp\big(\mathbb E[\log\theta_{dk}]+\mathbb E[\log\beta_{kw}]\big),\qquad
\gamma_{dk}=\alpha+\sum_w x_{dw}\,\phi_{dwk},\qquad
\hat\lambda_{kw}=\eta+\frac{D}{S}\sum_{d\in\text{batch}}x_{dw}\,\phi_{dwk}.
$$

**Data.** 20 Newsgroups preprocessed by OCTIS (lemmatised, stop words removed, vocabulary of 1612 words; file `data/20newsgroups_octis/`). We use the training documents of six newsgroups.

In [ ]:
DATA_DIR = "1E/data/20newsgroups_octis/"
corpus = pd.read_csv(DATA_DIR + "corpus.tsv", sep="\t", header=None, names=["text", "split", "label"], quoting=3)
vocab = open(DATA_DIR + "vocabulary.txt").read().split()
word_index = {word: i for i, word in enumerate(vocab)}
groups = ["comp.graphics", "rec.sport.hockey", "sci.space", "sci.med", "talk.politics.mideast", "soc.religion.christian"]
docs = corpus[corpus["label"].isin(groups) & (corpus["split"] == "train")].reset_index(drop=True)


def bag_of_words(texts, word_index):
    # Sparse D x W count matrix.
    rows, cols, vals = [], [], []
    for d, text in enumerate(texts):
        ids, counts = np.unique([word_index[t] for t in text.split() if t in word_index], return_counts=True)
        rows.extend([d] * len(ids)); cols.extend(ids); vals.extend(counts)
    return sps.csr_matrix((vals, (rows, cols)), shape=(len(texts), len(word_index)), dtype=float)


X_bow = bag_of_words(docs["text"], word_index)
labels_bow = docs["label"].to_numpy()
doc_lengths = np.asarray(X_bow.sum(axis=1)).ravel()
print(f"D = {X_bow.shape[0]} documents, W = {X_bow.shape[1]} words, {int(doc_lengths.sum())} tokens")
print(f"document length: min {doc_lengths.min():.0f}, median {np.median(doc_lengths):.0f}, max {doc_lengths.max():.0f}")


def calculate_elbo_bow(X, lmbda, alpha, eta, local_step, chunk=1000):
    # Provided: full ELBO for bag-of-words LDA. The local parameters of all documents are refitted with your
    # local_step for the current lambda (in chunks, to save memory).
    D, W = X.shape
    K = lmbda.shape[0]
    E_log_beta = expected_log_dirichlet(lmbda)
    elbo = 0.0
    for start in range(0, D, chunk):
        Xc = X[start:start + chunk].tocoo()
        gamma, phi = local_step(Xc, E_log_beta, alpha)
        d, w, c = Xc.row, Xc.col, Xc.data
        E_log_theta = expected_log_dirichlet(gamma)
        elbo += np.sum(c[:, None] * (phi * (E_log_theta[d] + E_log_beta[:, w].T) - sp_spec.xlogy(phi, phi)))
        elbo += np.sum((alpha - gamma) * E_log_theta) + np.sum(sp_spec.gammaln(gamma) - sp_spec.gammaln(alpha)) \
            + np.sum(sp_spec.gammaln(K * alpha) - sp_spec.gammaln(gamma.sum(axis=1)))
    elbo += np.sum((eta - lmbda) * E_log_beta) + np.sum(sp_spec.gammaln(lmbda) - sp_spec.gammaln(eta)) \
        + np.sum(sp_spec.gammaln(W * eta) - sp_spec.gammaln(lmbda.sum(axis=1)))
    return elbo


def top_words(lmbda, vocab, n=10):
    return [[vocab[i] for i in np.argsort(-lmbda[k])[:n]] for k in range(lmbda.shape[0])]

Implement `local_step_bow` and `SVI_bow`.
- `local_step_bow(Xb, E_log_beta, alpha)` receives a batch as a sparse COO matrix `Xb` ($S\times W$) with entries `Xb.row` (document within the batch), `Xb.col` (word id) and `Xb.data` (count). It iterates the $\phi$ and $\gamma$ updates to convergence and returns `gamma` $[S,K]$ and `phi` $[\text{nnz},K]$, one row per non-zero entry of `Xb`. Hint: a sparse matrix, or `np.add.at`, sums over the entries of each document.
- `SVI_bow` runs mini-batch SVI with the global update above for `n_epochs` epochs of $\lfloor D/S\rfloor$ steps, and records the ELBO per word after every epoch using `calculate_elbo_bow` (the evaluation is not timed).

Run it with $K=6$, $\alpha=1/K$, $\eta=0.01$, $S=128$, $\tau=1$, $\kappa=0.7$ for 15 epochs. Report the ELBO per word over epochs and the runtime, the 10 most probable words of every topic, and a heat map of the average $\mathbb E_q[\theta_d]$ per newsgroup. Comment briefly on the topics.

In [ ]:
def local_step_bow(Xb, E_log_beta, alpha, max_iter=100, tol=1e-3):
    # Insert your code here
    return gamma, phi


def SVI_bow(X, K, S, n_epochs, alpha, eta, rng, tau=1.0, kappa=0.7):
    # Returns lmbda [K, W], ELBO per word per epoch, and the runtime excluding ELBO evaluation.
    D, W = X.shape
    lmbda = rng.gamma(100.0, 0.01, size=(K, W))    # random initialisation around 1
    n_tokens = X.sum()
    elbo_per_word, train_time, t = [], 0.0, 0
    for epoch in tqdm.tqdm(range(n_epochs), desc="SVI bag-of-words"):
        t0 = time.time()
        for _ in range(D // S):
            # TODO: sample a batch, run the local step, compute lambda_hat and take the natural-gradient step
            t += 1
        train_time += time.time() - t0
        elbo_per_word.append(calculate_elbo_bow(X, lmbda, alpha, eta, local_step_bow) / n_tokens)
    return lmbda, np.array(elbo_per_word), train_time

In [ ]:
rng4 = np.random.default_rng(4)  # Do not change the seed
K4 = len(groups)
# Run SVI_bow and produce the requested outputs

*Your comments here.*